# microgpt + tiny-ton

This notebook runs [Karpathy's microgpt](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95) (pure Python, zero-dependency GPT training) and sets up **tiny-ton** alongside it.

The goal: profile where microgpt spends its time, then incrementally replace the hot operations with tiny-ton JIT-compiled GPU kernels.

**Before running:** Go to *Runtime > Change runtime type* and select **T4 GPU**.

## 1. Download microgpt.py

In [1]:
!wget -q https://gist.githubusercontent.com/karpathy/8627fe009c40f57531cb18360106ce95/raw/microgpt.py -O /content/microgpt.py
!head -6 /content/microgpt.py

"""
The most atomic way to train and run inference for a GPT in pure, dependency-free Python.
This file is the complete algorithm.
Everything else is just efficiency.

@karpathy


## 2. Run microgpt (baseline)

Pure Python, no dependencies, no GPU. Trains a 1-layer GPT on character-level name generation.

Default: `n_layer=1, n_embd=16, block_size=16, n_head=4, 1000 steps`.

This takes ~1-2 minutes on Colab's CPU.

In [2]:
%%time
!cd /content && python3 microgpt.py

num docs: 32033
vocab size: 27
num params: 4192
step 1000 / 1000 | loss 2.6497
--- inference (new, hallucinated names) ---
sample  1: kamon
sample  2: ann
sample  3: karai
sample  4: jaire
sample  5: vialan
sample  6: karia
sample  7: yeran
sample  8: anna
sample  9: areli
sample 10: kaina
sample 11: konna
sample 12: keylen
sample 13: liole
sample 14: alerin
sample 15: earan
sample 16: lenne
sample 17: kana
sample 18: lara
sample 19: alela
sample 20: anton
CPU times: user 1.37 s, sys: 269 ms, total: 1.64 s
Wall time: 4min 59s


## 3. Profile: where does microgpt spend its time?

Run 10 training steps under `cProfile` to see which functions dominate.

In [3]:
%%writefile /content/profile_microgpt.py
"""Profile microgpt to find the hot path."""
import cProfile
import pstats
import os
import math
import random
random.seed(42)

# --- Dataset ---
if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')
docs = [line.strip() for line in open('input.txt') if line.strip()]
random.shuffle(docs)

# --- Tokenizer ---
uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

# --- Autograd ---
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')
    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))
    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

# --- Model ---
n_layer = 1; n_embd = 16; block_size = 16; n_head = 4
head_dim = n_embd // n_head
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
params = [p for mat in state_dict.values() for row in mat for p in row]

def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id]
    pos_emb = state_dict['wpe'][pos_id]
    x = [t + p for t, p in zip(tok_emb, pos_emb)]
    x = rmsnorm(x)
    for li in range(n_layer):
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)
        values[li].append(v)
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]
    logits = linear(x, state_dict['lm_head'])
    return logits

learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params)
v = [0.0] * len(params)

# Profile just 10 steps
num_steps = 10

def train_loop():
    for step in range(num_steps):
        doc = docs[step % len(docs)]
        tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
        n = min(block_size, len(tokens) - 1)
        keys_buf, values_buf = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
        losses = []
        for pos_id in range(n):
            token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
            logits = gpt(token_id, pos_id, keys_buf, values_buf)
            probs = softmax(logits)
            loss_t = -probs[target_id].log()
            losses.append(loss_t)
        loss = (1 / n) * sum(losses)
        loss.backward()
        lr_t = learning_rate * (1 - step / num_steps)
        for i, p in enumerate(params):
            m[i] = beta1 * m[i] + (1 - beta1) * p.grad
            v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
            m_hat = m[i] / (1 - beta1 ** (step + 1))
            v_hat = v[i] / (1 - beta2 ** (step + 1))
            p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
            p.grad = 0
        print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}")

profiler = cProfile.Profile()
profiler.enable()
train_loop()
profiler.disable()

print("\n" + "="*70)
print("  PROFILE: Top 20 functions by cumulative time")
print("="*70)
stats = pstats.Stats(profiler)
stats.strip_dirs().sort_stats('cumulative').print_stats(20)

print("="*70)
print("  PROFILE: Top 20 functions by total time (self)")
print("="*70)
stats.strip_dirs().sort_stats('tottime').print_stats(20)

Writing /content/profile_microgpt.py


In [4]:
!cd /content && python3 profile_microgpt.py

step    1 /   10 | loss 3.3660
step    2 /   10 | loss 3.4243
step    3 /   10 | loss 3.1774
step    4 /   10 | loss 3.0726
step    5 /   10 | loss 3.2317
step    6 /   10 | loss 3.0026
step    7 /   10 | loss 3.3227
step    8 /   10 | loss 3.3149
step    9 /   10 | loss 3.0019
step   10 /   10 | loss 3.2534

  PROFILE: Top 20 functions by cumulative time
         4421151 function calls (3301309 primitive calls) in 3.875 seconds

   Ordered by: cumulative time
   List reduced from 37 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.141    0.141    3.875    3.875 profile_microgpt.py:131(train_loop)
       73    0.011    0.000    2.313    0.032 profile_microgpt.py:90(gpt)
    15493    0.105    0.000    2.180    0.000 {built-in method builtins.sum}
      511    0.011    0.000    2.133    0.004 profile_microgpt.py:76(linear)
   588950    1.554    0.000    1.554    0.000 profile_microgpt.py:25(__init__)
   268275    0.096 

## 4. Install LLVM/MLIR 18 + build tiny-ton

Set up tiny-ton so we can start replacing hot operations with JIT-compiled GPU kernels.

In [5]:
%%bash
set -e
echo '>>> Adding LLVM 18 apt repository...'
wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key | tee /etc/apt/trusted.gpg.d/apt.llvm.org.asc > /dev/null
add-apt-repository -y 'deb http://apt.llvm.org/jammy/ llvm-toolchain-jammy-18 main' > /dev/null 2>&1
echo '>>> Installing packages...'
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq libmlir-18-dev mlir-18-tools llvm-18-dev cmake ninja-build > /dev/null 2>&1
pip install -q pybind11
echo '>>> Done. MLIR version:'
mlir-opt-18 --version 2>&1 | head -2

>>> Adding LLVM 18 apt repository...
>>> Installing packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 7.1 MB/s eta 0:00:00
>>> Done. MLIR version:
Ubuntu LLVM version 18.1.8
  Optimized build.


In [6]:
%%bash
set -e
if [ ! -d /content/tiny-ton ]; then
  git clone https://github.com/ganeshnj/tiny-ton.git /content/tiny-ton
else
  echo 'tiny-ton already cloned, pulling latest...'
  cd /content/tiny-ton && git pull
fi

Cloning into '/content/tiny-ton'...


In [7]:
%%bash
set -e
cd /content/tiny-ton
rm -rf build
cmake -G Ninja -S . -B build \
  -DCMAKE_BUILD_TYPE=Release \
  -DMLIR_DIR=/usr/lib/llvm-18/lib/cmake/mlir \
  -DLLVM_DIR=/usr/lib/llvm-18/lib/cmake/llvm \
  -DTTN_ENABLE_CUDA=ON \
  -DTTN_ENABLE_PYTHON=ON \
  -DCUDAToolkit_ROOT=/usr/local/cuda \
  -Dpybind11_DIR=$(python3 -c 'import pybind11; print(pybind11.get_cmake_dir())')
cmake --build build
echo '>>> Build complete.'

-- The CXX compiler identification is GNU 11.4.0
-- The C compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Performing Test HAVE_FFI_CALL
-- Performing Test HAVE_FFI_CALL - Success
-- Found FFI: /usr/lib/x86_64-linux-gnu/libffi.so
-- Could NOT find LibEdit (missing: LibEdit_INCLUDE_DIRS LibEdit_LIBRARIES) 
-- Performing Test Terminfo_LINKABLE
-- Performing Test Terminfo_LINKABLE - Success
-- Found Terminfo: /usr/lib/x86_64-linux-gnu/libtinfo.so
-- Found ZLIB: /usr/lib/x86_64-linux-gnu/libz.so (found version "1.2.11")
-- Found zstd: /usr/lib/x86_64-linux-gnu/libzstd.so
-- Found LibXml2: /usr/li

CMake Error at /usr/local/lib/python3.12/dist-packages/cmake/data/share/cmake-3.31/Modules/FindCUDAToolkit.cmake:908 (message):
  Could not find nvcc, please set CUDAToolkit_ROOT.
Call Stack (most recent call first):
  CMakeLists.txt:34 (find_package)




CalledProcessError: Command 'b"set -e\ncd /content/tiny-ton\ncmake -G Ninja -S . -B build \\\n  -DCMAKE_BUILD_TYPE=Release \\\n  -DMLIR_DIR=/usr/lib/llvm-18/lib/cmake/mlir \\\n  -DLLVM_DIR=/usr/lib/llvm-18/lib/cmake/llvm \\\n  -DTTN_ENABLE_CUDA=ON \\\n  -DTTN_ENABLE_PYTHON=ON \\\n  -Dpybind11_DIR=$(python3 -c 'import pybind11; print(pybind11.get_cmake_dir())')\ncmake --build build\necho '>>> Build complete.'\n"' returned non-zero exit status 1.

In [ ]:
import sys, os
sys.path.insert(0, '/content/tiny-ton/build/bindings')
sys.path.insert(0, '/content/tiny-ton/python')
os.environ['TTN_SM_VERSION'] = 'sm_75'

import numpy as np
import tiny_ton as tt
import _tiny_ton_core as core

print(f'has_cuda() = {core.has_cuda()}')
!nvidia-smi -L

## 5. tiny-ton vector_add on the T4 GPU

Verify tiny-ton works: run vector_add with f32 on the real GPU.

In [ ]:
@tt.jit
def vector_add(a_ptr, b_ptr, c_ptr, N):
    pid = tt.program_id(0)
    offsets = pid * 64 + tt.arange(0, 64)
    mask = offsets < N
    a = tt.load(a_ptr + offsets, mask=mask)
    b = tt.load(b_ptr + offsets, mask=mask)
    tt.store(c_ptr + offsets, a + b, mask=mask)

N = 1024
for dtype_name, dtype in [('i32', np.int32), ('f32', np.float32), ('f16', np.float16)]:
    a = np.random.randn(N).astype(dtype)
    b = np.random.randn(N).astype(dtype)
    c = np.zeros(N, dtype=dtype)
    vector_add[(N // 64,)](a, b, c, N)
    expected = (a + b).astype(dtype)
    atol = 1e-2 if dtype == np.float16 else 1e-6 if dtype == np.float32 else 0
    ok = np.allclose(c, expected, atol=atol) if atol else np.array_equal(c, expected)
    print(f'{dtype_name}: {"PASS" if ok else "FAIL"}')

## 5b. Math intrinsics on the T4 GPU

Test the building blocks of softmax and rmsnorm: `exp`, `log`, `sqrt`, `rsqrt`, `abs`, `max`.

In [ ]:
@tt.jit
def kern_exp(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.exp(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_log(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.log(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_sqrt(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.sqrt(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_rsqrt(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.rsqrt(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_abs(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.abs(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_max(a_ptr, b_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    a = tt.load(a_ptr + off, mask=mask)
    b = tt.load(b_ptr + off, mask=mask)
    tt.store(dst + off, tt.max(a, b), mask=mask)

N = 256
grid = (N // 64,)
np.random.seed(42)

for dtype_name, dtype, atol in [('f32', np.float32, 1e-5), ('f16', np.float16, 5e-2)]:
    print(f'\n--- {dtype_name} ---')
    pos = (np.abs(np.random.randn(N)) + 0.01).astype(dtype)
    mixed = (np.random.randn(N) * 2).astype(dtype)

    for name, kernel, np_fn, inp in [
        ('exp',  kern_exp,  np.exp,  mixed),
        ('log',  kern_log,  np.log,  pos),
        ('sqrt', kern_sqrt, np.sqrt, pos),
        ('abs',  kern_abs,  np.abs,  mixed),
    ]:
        dst = np.zeros(N, dtype=dtype)
        kernel[grid](inp.copy(), dst, N)
        ok = np.allclose(dst, np_fn(inp), atol=atol, rtol=1e-2)
        print(f'  {name}: {"PASS" if ok else "FAIL"}')

    # rsqrt
    dst = np.zeros(N, dtype=dtype)
    kern_rsqrt[grid](pos.copy(), dst, N)
    expected_rsqrt = (1.0 / np.sqrt(pos.astype(np.float32))).astype(dtype)
    ok = np.allclose(dst, expected_rsqrt, atol=atol, rtol=1e-2)
    print(f'  rsqrt: {"PASS" if ok else "FAIL"}')

    # max (binary)
    a = (np.random.randn(N) * 5).astype(dtype)
    b = (np.random.randn(N) * 5).astype(dtype)
    dst = np.zeros(N, dtype=dtype)
    kern_max[grid](a.copy(), b.copy(), dst, N)
    ok = np.allclose(dst, np.maximum(a, b), atol=atol, rtol=1e-2)
    print(f'  max: {"PASS" if ok else "FAIL"}')

print('\nAll math intrinsics ready for softmax/rmsnorm!')

## 6. The road ahead: microgpt x tiny-ton

The profile above shows where microgpt spends its time. The hot functions map to GPU kernels:

| microgpt function | Compute | tiny-ton status | What's needed |
|---|---|---|---|
| `linear(x, w)` | matrix-vector multiply | -- | tiled matmul, shared memory |
| `softmax(logits)` | exp, sum, div | **partial** (exp, max done) | reduction (`sum`) |
| `rmsnorm(x)` | square, sum, rsqrt, scale | **partial** (sqrt, rsqrt done) | reduction (`sum`) |
| `Value.__add__` | element-wise add | **done** (i32/f32/f16) | -- |
| `Value.__mul__` | element-wise mul | **done** (i32/f32/f16) | -- |
| `Value.exp()` | element-wise exp | **done** (f32/f16) | -- |
| `Value.log()` | element-wise log | **done** (f32/f16) | -- |
| `Value.backward()` | autograd chain rule | -- | user writes backward kernels |

### Math intrinsics now available:

`tt.exp`, `tt.log`, `tt.sqrt`, `tt.rsqrt`, `tt.abs`, `tt.max` -- all work on f32 and f16.

### Next steps:

1. **Reductions** (`sum`, `max` over a vector) -- enables softmax, rmsnorm, loss
2. **Tiled matmul** with shared memory -- enables `linear()`, the biggest bottleneck
3. **Fused kernels** -- combine multiple ops into one GPU launch